# Step 1: Environment Setup & Dataset Download
## Attention-CNN-LSTM for Intrusion Detection (Bot-IoT Dataset)

This notebook will:
1. Verify your environment (Python, PyTorch, CUDA)
2. Install all required libraries
3. Guide you to download the Bot-IoT dataset
4. Do a quick sanity check on the data

> **Run each cell one by one. Read the output before moving to the next.**

---
## Cell 1: Check Python Version
We need Python 3.8 or higher.

In [1]:
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 8), "Please upgrade to Python 3.8 or higher!"
print("✅ Python version is OK")

Python version: 3.12.13 (main, May 10 2026, 19:35:37) [MSC v.1944 64 bit (AMD64)]
✅ Python version is OK


---
## Cell 2: Install Required Libraries
Run this cell to install everything needed for the project.

> **Note:** If you already have these installed, this will just confirm versions. It may take 2-3 minutes.

In [2]:
# Install all required packages
# Run this cell once. After installation, you won't need to run it again.

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 -q
!pip install pandas numpy scikit-learn matplotlib seaborn tqdm -q

print("\n✅ All packages installed successfully")

ERROR: Could not find a version that satisfies the requirement torch (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BA\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch



✅ All packages installed successfully



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BA\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


---
## Cell 3: Verify PyTorch and CUDA (GPU)
This confirms your GPU is accessible by PyTorch.

In [3]:
import torch

print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name            : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version        : {torch.version.cuda}")
    print(f"GPU memory (GB)     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
    device = torch.device('cuda')
    print("\n✅ GPU is ready! Training will be fast.")
else:
    device = torch.device('cpu')
    print("\n⚠️  GPU not detected. Will use CPU (slower but works).")
    print("   If you have an NVIDIA GPU, make sure CUDA drivers are installed.")
    print("   Download: https://developer.nvidia.com/cuda-downloads")

print(f"\nUsing device: {device}")

PyTorch version     : 2.6.0+cu124
CUDA available      : True
GPU name            : NVIDIA GeForce GTX 1050
CUDA version        : 12.4
GPU memory (GB)     : 4.29

✅ GPU is ready! Training will be fast.

Using device: cuda


---
## Cell 4: Import All Libraries
If any import fails, re-run Cell 2 to reinstall.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("torch      :", torch.__version__)
print("sklearn    : OK")
print("matplotlib : OK")
print("\n✅ All imports successful!")

pandas     : 3.0.3
numpy      : 2.4.4
torch      : 2.6.0+cu124
sklearn    : OK
matplotlib : OK

✅ All imports successful!


---
## Cell 5: Create Project Folder Structure
This creates organized folders for your project.

In [5]:
import os

# Define folder structure
folders = [
    'data/raw',        # Raw downloaded dataset goes here
    'data/processed',  # Cleaned and preprocessed data
    'models',          # Saved model checkpoints
    'results',         # Plots, confusion matrices, metrics
    'notebooks'        # All your notebooks
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Created: {folder}/")

print("\n📁 Project structure ready!")
print("""
project/
├── data/
│   ├── raw/          ← Put downloaded Bot-IoT CSV files here
│   └── processed/    ← Cleaned data will be saved here
├── models/           ← Trained model will be saved here
├── results/          ← Plots and evaluation results
└── notebooks/        ← All notebooks
""")

✅ Created: data/raw/
✅ Created: data/processed/
✅ Created: models/
✅ Created: results/
✅ Created: notebooks/

📁 Project structure ready!

project/
├── data/
│   ├── raw/          ← Put downloaded Bot-IoT CSV files here
│   └── processed/    ← Cleaned data will be saved here
├── models/           ← Trained model will be saved here
├── results/          ← Plots and evaluation results
└── notebooks/        ← All notebooks



---
## Cell 6: Download Instructions for Bot-IoT Dataset

The Bot-IoT dataset is hosted by the University of New South Wales (UNSW).
It is free but requires manual download from their website.

### Steps to Download:

1. Go to: **https://research.unsw.edu.au/projects/bot-iot-dataset**
2. Scroll down and click **"Download Dataset"**
3. You will be asked to fill a short form (name, email, institution)
4. Download the **"CSV files (Feature Extracted)"** version — NOT the raw PCAP files
5. Extract the ZIP file
6. You will get multiple CSV files like:
   - `UNSW_2018_IoT_Botnet_Full5pc_1.csv`
   - `UNSW_2018_IoT_Botnet_Full5pc_2.csv`
   - ... (multiple parts)
7. **Copy ALL CSV files into the `data/raw/` folder** you just created above

> **Note:** The full dataset is ~2.5 million records. 
> For initial experimentation, we will use the **5% sample** files which are much more manageable.
> The 5% sample files are named: `UNSW_2018_IoT_Botnet_Full5pc_X.csv`

Run the next cell AFTER you have placed the CSV files in `data/raw/`

In [6]:
# Run this cell AFTER placing CSV files in data/raw/

raw_path = 'data/raw/'
csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]

if len(csv_files) == 0:
    print("❌ No CSV files found in data/raw/")
    print("   Please download the dataset and place CSV files in data/raw/")
    print("   Follow the instructions in Cell 6 above.")
else:
    print(f"✅ Found {len(csv_files)} CSV file(s) in data/raw/:")
    total_size = 0
    for f in csv_files:
        size_mb = os.path.getsize(os.path.join(raw_path, f)) / 1e6
        total_size += size_mb
        print(f"   📄 {f} ({size_mb:.1f} MB)")
    print(f"\n   Total size: {total_size:.1f} MB")

✅ Found 4 CSV file(s) in data/raw/:
   📄 UNSW_2018_IoT_Botnet_Dataset_1.csv (243.0 MB)
   📄 UNSW_2018_IoT_Botnet_Dataset_2.csv (243.2 MB)
   📄 UNSW_2018_IoT_Botnet_Dataset_3.csv (224.3 MB)
   📄 UNSW_2018_IoT_Botnet_Dataset_4.csv (224.8 MB)

   Total size: 935.3 MB


---
## Cell 7: Load and Inspect the Dataset
Let's take a first look at what the data looks like.

In [7]:
# Load all CSV files and combine them
raw_path = 'data/raw/'
csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]

print("Loading CSV files...")
dfs = []
for f in csv_files:
    print(f"  Loading {f}...")
    df_temp = pd.read_csv(os.path.join(raw_path, f), low_memory=False)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
print(f"\n✅ Dataset loaded!")
print(f"   Total rows    : {df.shape[0]:,}")
print(f"   Total columns : {df.shape[1]}")

Loading CSV files...
  Loading UNSW_2018_IoT_Botnet_Dataset_1.csv...
  Loading UNSW_2018_IoT_Botnet_Dataset_2.csv...
  Loading UNSW_2018_IoT_Botnet_Dataset_3.csv...
  Loading UNSW_2018_IoT_Botnet_Dataset_4.csv...

✅ Dataset loaded!
   Total rows    : 3,999,996
   Total columns : 93


In [8]:
# Preview the first few rows
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,1,1526344121.188091,e,arp,192.168.100.1,Unnamed: 5,192.168.100.3,Unnamed: 7,4,240,...,54919,3,462,1528081566.286314,91429,42.664722,3.1,462.1,0.046877,0.046877.1
0,2,1.526344e+09,e,tcp,192.168.100.7,139,192.168.100.4,36390,10.0,680.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,1.526344e+09,e,udp,192.168.100.149,51838,27.124.125.250,123,2.0,180.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,1.526344e+09,e,arp,192.168.100.4,NaN,192.168.100.7,NaN,10.0,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,1.526344e+09,e,udp,192.168.100.27,58999,192.168.100.1,53,4.0,630.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,1.526344e+09,e,arp,192.168.100.1,NaN,192.168.100.27,NaN,2.0,120.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Check column names
print("All column names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")

All column names:
   1. 1
   2. 1526344121.188091
   3. e
   4. arp
   5. 192.168.100.1
   6. Unnamed: 5
   7. 192.168.100.3
   8. Unnamed: 7
   9. 4
  10. 240
  11. CON
  12. 1526345317.184693
  13. 9
  14. 1195.996582
  15. 0.000006
  16. 0.000002
  17. Unnamed: 16
  18. Unnamed: 17
  19. 0.000011
  20. 0.000004
  21. 0.000007
  22. Unnamed: 21
  23. Unnamed: 22
  24. Unnamed: 23
  25. Unnamed: 24
  26. 2
  27. 2.1
  28. 120
  29. 120.1
  30. 0.002508
  31. 0.000836
  32. 0.000836.1
  33. 0
  34. Normal
  35. Normal.1
  36. 1000001
  37. 1526949252.645504
  38. tcp
  39. 192.168.100.150
  40. 49731
  41. 41081
  42. RST
  43. 1526949252.687410
  44. 180408
  45. 0.041906
  46. 0.041906.1
  47. 0.041906.2
  48. 0.041906.3
  49. 0.041906.4
  50. 1.1
  51. 60
  52. 60.1
  53. 23.862932
  54. 0.1
  55. 0.2
  56. 1.2
  57. Reconnaissance
  58. Service_Scan
  59. 2000001
  60. 1528081325.528940
  61. e s
  62. 192.168.100.149
  63. 30117
  64. 192.168.100.5
  65. 80
  66. 616
  67. REQ
  6

In [10]:
# Check data types and missing values
print("Data info:")
print(df.dtypes)
print(f"\nMissing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  No missing values! ✅")

Data info:
1                      int64
1526344121.188091    float64
e                        str
arp                      str
192.168.100.1            str
                      ...   
42.664722            float64
3.1                  float64
462.1                float64
0.046877             float64
0.046877.1           float64
Length: 93, dtype: object

Missing values per column:
1526344121.188091    2999997
e                    1999998
arp                  2999997
192.168.100.1        2999997
Unnamed: 5           3000483
                      ...   
42.664722            2999997
3.1                  2999997
462.1                2999997
0.046877             2999997
0.046877.1           2999997
Length: 91, dtype: int64


In [ ]:
# Check the target label distribution
# The label column is usually 'category' or 'attack' or 'label'
# Let's find it automatically

possible_label_cols = ['pkSeqID', 'Stime', 'Flgs', 'flgs_number', 'Proto', 'proto_number', 'Saddr', 'Sport', 'Daddr', 'Dport', 'Pkts', 'Bytes', 'State', 'state_number', 'Ltime', 'Seq',
                       'Dur', 'Mean', 'Stddev', 'Sum', 'Min', 'Max', 'Spkts', 'Dpkts', 'Sbytes', 'Dbytes', 'Rate', 'Srate', 'Drate', 'TnBPSrcIP', 'TnBPDstIP', 'TnP_PSrcIP', 'TnP_PDstIP', 
                    'TnP_PerProto', 'TnP_Per_Dport', 'AR_P_Proto_P_SrcIP', 'AR_P_Proto_P_DstIP', 'N_IN_Conn_P_SrcIP', 'N_IN_Conn_P_DstIP', 'AR_P_Proto_P_Sport', 'AR_P_Proto_P_Dport',
                    'Pkts_P_State_P_Protocol_P_DestIP', 'Pkts_P_State_P_Protocol_P_SrcIP', 'Attack', 'Category', 'Subcategory']
found_labels = [col for col in possible_label_cols if col in df.columns]

print("Potential label columns found:", found_labels)

if found_labels:
    for col in found_labels:
        print(f"\nValue counts for '{col}':")
        print(df[col].value_counts())
        print(f"Unique values: {df[col].nunique()}")

In [ ]:
# Visualize class distribution
# Update 'category' below if your label column has a different name
LABEL_COL = 'category'  # <-- Change this if needed based on output above

if LABEL_COL in df.columns:
    plt.figure(figsize=(10, 5))
    df[LABEL_COL].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('Class Distribution in Bot-IoT Dataset', fontsize=14)
    plt.xlabel('Attack Category')
    plt.ylabel('Number of Records')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('results/class_distribution.png', dpi=150)
    plt.show()
    print("✅ Plot saved to results/class_distribution.png")
else:
    print(f"❌ Column '{LABEL_COL}' not found. Update LABEL_COL variable above.")

---
## Cell 8: Save Dataset Info for Next Notebook

In [ ]:
# Save column names and basic info for reference in next notebooks
info = {
    'total_rows': df.shape[0],
    'total_cols': df.shape[1],
    'columns': list(df.columns),
    'label_col': LABEL_COL,
    'classes': list(df[LABEL_COL].unique()) if LABEL_COL in df.columns else []
}

import json
with open('data/processed/dataset_info.json', 'w') as f:
    json.dump(info, f, indent=2)

print("✅ Dataset info saved to data/processed/dataset_info.json")
print("\n" + "="*50)
print("SETUP COMPLETE! You are ready for Notebook 2.")
print("="*50)
print("""
Summary:
  ✅ Python & PyTorch verified
  ✅ All libraries installed
  ✅ Project folders created
  ✅ Dataset loaded and inspected
  ✅ Class distribution visualized

Next step: Run notebook  →  02_preprocessing.ipynb
""")

---
## Troubleshooting Guide

| Problem | Solution |
|--------|----------|
| `ModuleNotFoundError` | Re-run Cell 2 to reinstall libraries |
| `CUDA not available` | Download CUDA from https://developer.nvidia.com/cuda-downloads |
| CSV files not found | Make sure files are in `data/raw/` folder, not a subfolder |
| Label column not found | Check Cell 7 output and update `LABEL_COL` variable |
| Memory error on loading | Use only 1-2 CSV files instead of all of them |